In [ ]:
# %%
# ============================================================================
# Figure 7 — Temporal forecasting diagnostics (RIFT manuscript)
#
#   (A-C) Predicted-vs-actual DTW on the temporal test set, classified
#         'Good' (|error| <= 5 m, green) or 'Poor' (> 5 m, red),
#         for Extra Trees, Random Forest, and Decision Tree.
#   (D)   Spatial distribution of aggregated well performance across Chile
#         (Good / Poor / Both) for the same three models.
#
# Composite of the former Figure 21 (Good/Poor scatter) and Figure 22 (maps).
#
# *** SINGLE SOURCE OF TRUTH ***
# Everything (model, lon, lat, actual, predicted, category) comes from ONE CSV.
# The per-record rows feed the scatters directly; the maps are built by
# aggregating each well's many records into one Good / Poor / Both label.
#
# Map rendering uses geopandas + your basin shapefile when available, and falls
# back to a self-contained Chile outline so the figure still previews without
# those assets. Jupyter-ready: run as one cell or split on the "# %%" markers.
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from pathlib import Path

# Optional geospatial stack (only needed for real basin shapefile / hillshade)
try:
    import geopandas as gpd
    _HAS_GPD = True
except Exception:
    _HAS_GPD = False

# %%
# --- Publication styling -----------------------------------------------------
plt.rcParams.update({
    "font.family":        ["Arial", "DejaVu Sans", "sans-serif"],
    "font.size":          10,
    "axes.titlesize":     11,
    "axes.labelsize":     10,
    "xtick.labelsize":    8.5,
    "ytick.labelsize":    8.5,
    "legend.fontsize":    9.5,
    "axes.linewidth":     0.8,
    "figure.dpi":         150,
    "savefig.dpi":        600,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
})

MODEL_ORDER = ["Extra Trees", "Random Forest", "Decision Tree"]
PANEL_LETTER = {"Extra Trees": "A", "Random Forest": "B", "Decision Tree": "C"}

# Colours
C_GOOD, C_POOR, C_BOTH = "#2E7D32", "#D32F2F", "#E7A9BC"   # scatter + map categories
C_LAND, C_OCEAN        = "#f1efe9", "#dbe7f3"             # map land / ocean fill
C_BASIN                = "0.45"                            # basin boundary colour
ERR_THRESHOLD          = 5.0                               # m, Good/Poor cut-off

OUTPUT_PATH = Path(r"C:\Users\m20000907\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\draft_for_AI_model\figures_7_26\Figure7_temporal_diagnostics.png")

# %%
# --- Data + asset paths  (EDIT to your file) ---------------------------------
# ONE CSV with all per-record predictions. Used for BOTH the scatters (A-C) and,
# after well-level aggregation, the maps (D).
DATA_CSV = Path(r"C:\Users\m20000907\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\python_code\groundwater_analysis_results_20250711_114958\detailed_testing_results.csv")

# --- Column names in your CSV  (EDIT here if a header differs) ----------------
COL_MODEL  = "Model"                       # model name (must match MODEL_ORDER)
COL_WELL   = "Code"                        # unique well / station identifier
COL_LON    = "Longitude_GCS_WGS_1984"
COL_LAT    = "Latitude_GCS_WGS_1984"
COL_ACTUAL = "Depth to water (m)"          # measured DTW (the 'actual' axis)
COL_PRED   = "predicted"                   # model prediction
COL_CAT    = "Performance_Category"        # 'Good' / 'Poor'; derived if absent

# Geographic assets (optional). If BASINS_SHP is set and geopandas is installed,
# basins are drawn; otherwise the embedded outline below is used.
BASINS_SHP    = Path("basins/chile_basins.shp")   # e.g. DGA / BNA basin polygons
HILLSHADE_IMG = None  # optional pre-rendered background PNG aligned to MAP_EXTENT,
                      # shown via imshow(extent=MAP_EXTENT). Leave None for a clean fill.

# Map extent [lon_min, lon_max, lat_min, lat_max]. None -> derived from the data.
MAP_EXTENT = None

# %%
# --- Embedded Chile outline (fallback only; replaced by your basin shapefile) -
CHILE_OUTLINE_FALLBACK = [
    (-68.22,-21.494), (-67.828,-22.873), (-67.107,-22.736), (-66.985,-22.986), (-67.328,-24.025), (-68.418,-24.519),
    (-68.386,-26.185), (-68.595,-26.507), (-68.296,-26.899), (-69.001,-27.521), (-69.656,-28.459), (-70.014,-29.368),
    (-69.919,-30.336), (-70.535,-31.365), (-70.074,-33.091), (-69.815,-33.274), (-69.817,-34.194), (-70.388,-35.17),
    (-70.365,-36.005), (-71.122,-36.658), (-71.119,-37.577), (-70.815,-38.553), (-71.414,-38.916), (-71.681,-39.808),
    (-71.916,-40.832), (-71.747,-42.051), (-72.149,-42.255), (-71.915,-43.409), (-71.464,-43.788), (-71.794,-44.207),
    (-71.33,-44.408), (-71.223,-44.784), (-71.659,-44.974), (-71.552,-45.561), (-71.917,-46.885), (-72.447,-47.739),
    (-72.331,-48.244), (-72.648,-48.879), (-73.415,-49.318), (-73.328,-50.379), (-72.976,-50.741), (-72.31,-50.677),
    (-72.329,-51.426), (-71.915,-52.009), (-69.498,-52.143), (-68.572,-52.299), (-69.461,-52.292), (-69.943,-52.538),
    (-70.845,-52.899), (-71.006,-53.833), (-71.43,-53.856), (-72.558,-53.531), (-73.703,-52.835), (-73.703,-52.835),
    (-74.947,-52.263), (-75.26,-51.629), (-74.977,-51.043), (-75.48,-50.378), (-75.608,-48.674), (-75.183,-47.712),
    (-74.127,-46.939), (-75.644,-46.648), (-74.692,-45.764), (-74.352,-44.103), (-73.24,-44.455), (-72.718,-42.383),
    (-73.389,-42.118), (-73.701,-43.366), (-74.332,-43.225), (-74.018,-41.795), (-73.677,-39.942), (-73.218,-39.259),
    (-73.506,-38.283), (-73.588,-37.156), (-73.167,-37.124), (-72.553,-35.509), (-71.862,-33.909), (-71.438,-32.419),
    (-71.669,-30.921), (-71.37,-30.096), (-71.49,-28.861), (-70.905,-27.64), (-70.725,-25.706), (-70.404,-23.629),
    (-70.091,-21.393), (-70.164,-19.756), (-70.373,-18.348), (-69.858,-18.093), (-69.59,-17.58), (-69.1,-18.26),
    (-68.967,-18.982), (-68.442,-19.405), (-68.757,-20.373), (-68.22,-21.494),
]

# %%
# --- Demo data (used only if DATA_CSV is absent) -----------------------------
# Synthetic stand-in with the SAME shape as your real file: many records per
# well, one row each. Approximate, NOT real results.
#   >>> Replace by supplying DATA_CSV. <<<
def _demo_data(seed=7):
    rng = np.random.default_rng(seed)
    n_wells = 220
    # Well latitudes: central Chile heavy, sparse far north/south.
    lat = np.concatenate([
        rng.normal(-33.0, 2.2, int(n_wells * 0.55)),
        rng.normal(-29.0, 2.0, int(n_wells * 0.22)),
        rng.normal(-20.0, 1.6, int(n_wells * 0.10)),
        rng.normal(-38.5, 2.2, int(n_wells * 0.13)),
    ])
    lat = np.clip(lat, -42.0, -18.0)
    n_wells = lat.size
    # Keep wells within the narrow Chilean monitored strip (coast -> Andes foothills).
    lon_center = np.interp(lat, [-42, -36, -33, -30, -24, -18],
                                [-72.8, -72.0, -71.0, -70.8, -69.8, -69.6])
    lon = lon_center + rng.normal(0, 0.45, n_wells)
    codes = 5000000 + np.arange(n_wells)

    frames = []
    base_noise = {"Extra Trees": 3.0, "Random Forest": 3.6, "Decision Tree": 4.2}
    for model in MODEL_ORDER:
        for w in range(n_wells):
            n_rec = int(rng.integers(8, 40))                       # records per well
            actual = np.clip(rng.gamma(1.6, 9.0, n_rec), 0.01, 145)  # right-skewed DTW
            central = -35.5 < lat[w] < -29.5                      # stressed central band
            noise = base_noise[model] * (1.7 if central else 1.0)
            pred = actual + rng.normal(0, noise, n_rec) + rng.normal(0, actual * 0.04)
            pred = np.clip(pred, 0, None)
            frames.append(pd.DataFrame({
                COL_MODEL: model, COL_WELL: codes[w],
                "Longitude": lon[w], "Latitude": lat[w],
                "actual": actual, "predicted": pred,
            }))
    return pd.concat(frames, ignore_index=True)


# %%
# --- Load + standardise ------------------------------------------------------
def load_data():
    if DATA_CSV.exists():
        df = pd.read_csv(DATA_CSV)                 # tab-delimited? add sep="\t"
        df = df.rename(columns={COL_LON: "Longitude", COL_LAT: "Latitude",
                                COL_ACTUAL: "actual", COL_PRED: "predicted"})
        print(f"[ok]   loaded {len(df):,} records from {DATA_CSV}")
    else:
        print(f"[warn] {DATA_CSV} not found -- using SYNTHETIC demo data.\n"
              f"       Replace with your real CSV before publishing.")
        df = _demo_data()

    # Derive Good/Poor from the |error| threshold only if the column is absent.
    if COL_CAT not in df.columns:
        err = (df["predicted"] - df["actual"]).abs()
        df[COL_CAT] = np.where(err <= ERR_THRESHOLD, "Good", "Poor")
    # Normalise category labels (guards against stray whitespace like " Good").
    df[COL_CAT] = df[COL_CAT].astype(str).str.strip()
    return df


def aggregate_wells(df):
    """Collapse per-record predictions to ONE label per well & model.

    A well is 'Both' when its records contain a mix of Good and Poor; otherwise
    it is wholly 'Good' or wholly 'Poor'. One row per (model, well)."""
    def classify(s):
        n_good = int((s == "Good").sum())
        n_poor = int((s == "Poor").sum())
        if n_good and n_poor:
            return "Both"
        return "Poor" if n_poor else "Good"

    agg = (df.groupby([COL_MODEL, COL_WELL], as_index=False)
             .agg(Longitude=("Longitude", "first"),
                  Latitude=("Latitude", "first"),
                  Category=(COL_CAT, classify)))
    return agg


detailed = load_data()
wells    = aggregate_wells(detailed)

# Quick sanity readout for the manuscript (records vs. unique wells per model)
print("\nWells per category (aggregated):")
print(wells.groupby([COL_MODEL, "Category"]).size()
           .unstack(fill_value=0).reindex(MODEL_ORDER))

# Load basins once (if available)
basins = None
if _HAS_GPD and BASINS_SHP.exists():
    try:
        basins = gpd.read_file(BASINS_SHP).to_crs(4326)
        print(f"\n[ok]   loaded basins: {BASINS_SHP} ({len(basins)} polygons)")
    except Exception as e:
        print(f"\n[warn] could not read {BASINS_SHP} ({e}); using outline fallback.")

# Map extent
if MAP_EXTENT is None:
    pad = 0.6
    lon_c = 0.5 * (wells["Longitude"].min() + wells["Longitude"].max())
    lon_half = max(4.8, 0.5 * (wells["Longitude"].max() - wells["Longitude"].min()) + pad)
    MAP_EXTENT = [lon_c - lon_half, lon_c + lon_half,
                  wells["Latitude"].min() - pad, wells["Latitude"].max() + pad]
MEAN_LAT = float(np.mean(MAP_EXTENT[2:]))
MAP_ASPECT = 1.0 / np.cos(np.radians(abs(MEAN_LAT)))


# %%
# --- Helpers -----------------------------------------------------------------
def _metrics(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    ss_res = np.sum((actual - pred) ** 2)
    ss_tot = np.sum((actual - actual.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse = np.sqrt(np.mean((actual - pred) ** 2))
    mae = np.mean(np.abs(actual - pred))
    return r2, rmse, mae


def draw_scatter(ax, df, model):
    sub = df[df[COL_MODEL] == model]
    a, p = sub["actual"].values, sub["predicted"].values
    good = sub[COL_CAT].values == "Good"
    poor = ~good
    hi = max(a.max(), p.max()) * 1.04

    ax.fill_between([0, hi], [-ERR_THRESHOLD, hi - ERR_THRESHOLD],
                    [ERR_THRESHOLD, hi + ERR_THRESHOLD],
                    color="0.6", alpha=0.12, zorder=1, lw=0)
    ax.plot([0, hi], [0, hi], "k-", lw=1.2, zorder=4)
    ax.scatter(a[good], p[good], s=10, c=C_GOOD, alpha=0.55, edgecolors="none", zorder=3)
    ax.scatter(a[poor], p[poor], s=12, c=C_POOR, alpha=0.7, edgecolors="none", zorder=5)

    r2, rmse, mae = _metrics(a, p)
    n_good, n_poor = int(good.sum()), int(poor.sum())
    n = n_good + n_poor
    txt = (f"$R^2$ = {r2:.3f}\nRMSE = {rmse:.2f} m\nMAE = {mae:.2f} m\n"
           f"Good: {100*n_good/n:.1f}% (n={n_good:,})\n"
           f"Poor: {100*n_poor/n:.1f}% (n={n_poor:,})")
    ax.text(0.045, 0.955, txt, transform=ax.transAxes, fontsize=8.2, va="top",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.4", lw=0.8, alpha=0.95))

    ax.set_xlim(0, hi); ax.set_ylim(0, hi)
    ax.set_box_aspect(1)
    ax.grid(True, ls=":", lw=0.5, alpha=0.5)
    ax.set_xlabel("Actual DTW (m)")
    ax.set_title(f"({PANEL_LETTER[model]}) {model}", fontweight="bold")


def _north_arrow(ax):
    ax.annotate("N", xy=(0.085, 0.955), xytext=(0.085, 0.86), xycoords="axes fraction",
                ha="center", va="center", fontsize=11, fontweight="bold",
                arrowprops=dict(arrowstyle="-|>", color="k", lw=1.4))


def _scale_bar(ax, km=200):
    lon0, lon1, lat0, lat1 = MAP_EXTENT
    deg = km / (111.0 * np.cos(np.radians(abs(MEAN_LAT))))
    x0 = lon1 - 0.10 * (lon1 - lon0) - deg
    y0 = lat0 + 0.045 * (lat1 - lat0)
    ax.plot([x0, x0 + deg], [y0, y0], "k-", lw=2.6, solid_capstyle="butt", zorder=6)
    ax.text(x0 + deg / 2, y0 + 0.012 * (lat1 - lat0), f"{km} km",
            ha="center", va="bottom", fontsize=7.5, zorder=6)


def draw_map(ax, model, show_scale=False):
    lon0, lon1, lat0, lat1 = MAP_EXTENT
    ax.set_facecolor(C_OCEAN)
    if HILLSHADE_IMG is not None and Path(HILLSHADE_IMG).exists():
        ax.imshow(plt.imread(HILLSHADE_IMG), extent=MAP_EXTENT, origin="upper", zorder=0)
    if basins is not None:
        basins.plot(ax=ax, facecolor=C_LAND, edgecolor="none", zorder=1)
        basins.boundary.plot(ax=ax, color=C_BASIN, linewidth=0.45, zorder=2)
    else:
        ax.add_patch(MplPolygon(CHILE_OUTLINE_FALLBACK, closed=True,
                                facecolor=C_LAND, edgecolor=C_BASIN, lw=0.5, zorder=1))

    sub = wells[wells[COL_MODEL] == model]
    order = [("Good", C_GOOD, 14), ("Both", C_BOTH, 16), ("Poor", C_POOR, 18)]
    for cat, col, s in order:
        d = sub[sub["Category"] == cat]
        ax.scatter(d["Longitude"], d["Latitude"], s=s, c=col, edgecolors="k",
                   linewidths=0.3, alpha=0.95, zorder=5,
                   label=cat if model == MODEL_ORDER[0] else None)

    ax.set_xlim(lon0, lon1); ax.set_ylim(lat0, lat1)
    ax.set_aspect(MAP_ASPECT)
    ax.set_title(model, fontweight="bold", pad=6)
    ax.tick_params(length=2.5)
    ax.set_xticks(np.arange(np.ceil(lon0 / 3) * 3, lon1, 3))
    ax.set_yticks(np.arange(np.ceil(lat0 / 3) * 3, lat1, 3))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}\u00b0"))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}\u00b0"))
    for sp in ax.spines.values():
        sp.set_linewidth(0.8)
    _north_arrow(ax)
    if show_scale:
        _scale_bar(ax)


# %%
# --- Assemble the figure -----------------------------------------------------
fig = plt.figure(figsize=(11.5, 14.5))
outer = GridSpec(2, 1, height_ratios=[1.0, 2.55], hspace=0.22,
                 top=0.905, bottom=0.045, left=0.075, right=0.965)

gs_top = outer[0].subgridspec(1, 3, wspace=0.30)
gs_bot = outer[1].subgridspec(1, 3, wspace=0.10)

# Row A-C: scatters
scatter_axes = [fig.add_subplot(gs_top[0, i]) for i in range(3)]
for ax, model in zip(scatter_axes, MODEL_ORDER):
    draw_scatter(ax, detailed, model)
scatter_axes[0].set_ylabel("Predicted DTW (m)")

# Row D: maps (shared y)
map_axes = [fig.add_subplot(gs_bot[0, 0])]
map_axes += [fig.add_subplot(gs_bot[0, i], sharey=map_axes[0]) for i in (1, 2)]
for i, (ax, model) in enumerate(zip(map_axes, MODEL_ORDER)):
    draw_map(ax, model, show_scale=(i == 0))
    if i != 0:
        ax.tick_params(labelleft=False)
map_axes[0].set_ylabel("Latitude")
for ax in map_axes:
    ax.set_xlabel("Longitude")

# --- Group header for panel D ---
bb = map_axes[1].get_position()
fig.text(0.52, bb.y1 + 0.052, "(D)  Aggregated well performance across Chile",
         ha="center", fontsize=12, fontweight="bold")

# --- Shared legends ---
sc_handles = [
    Line2D([0], [0], marker="o", ls="none", mfc=C_GOOD, mec="none", ms=7, label="Good ($\\leq$5 m)"),
    Line2D([0], [0], marker="o", ls="none", mfc=C_POOR, mec="none", ms=7, label="Poor (>5 m)"),
    Line2D([0], [0], color="k", lw=1.2, label="1:1 line"),
    Patch(facecolor="0.6", alpha=0.25, label="$\\pm$5 m bounds"),
]
sc_bb = scatter_axes[1].get_position()
fig.legend(handles=sc_handles, loc="lower center", bbox_to_anchor=(0.52, sc_bb.y1 + 0.012),
           ncol=4, frameon=False, columnspacing=1.6, handletextpad=0.5)

map_handles = [
    Line2D([0], [0], marker="o", ls="none", mfc=C_GOOD, mec="k", mew=0.3, ms=7, label="Good"),
    Line2D([0], [0], marker="o", ls="none", mfc=C_POOR, mec="k", mew=0.3, ms=7, label="Poor"),
    Line2D([0], [0], marker="o", ls="none", mfc=C_BOTH, mec="k", mew=0.3, ms=7, label="Both"),
    Line2D([0], [0], color=C_BASIN, lw=0.8, label="Basin delimitation"),
]
fig.legend(handles=map_handles, loc="lower center", bbox_to_anchor=(0.52, bb.y1 + 0.030),
           ncol=4, frameon=False, columnspacing=1.6, handletextpad=0.5)

fig.savefig(OUTPUT_PATH, dpi=600, facecolor="white", bbox_inches="tight")
print(f"\n[ok]   saved: {OUTPUT_PATH.resolve()}")
plt.show()

# Version 2

In [ ]:
# %%
# ============================================================================
# Figure 7 — Temporal forecasting diagnostics (RIFT manuscript)
#
#   (A-C) Predicted-vs-actual DTW on the temporal test set, classified
#         'Good' (|error| <= 5 m, green) or 'Poor' (> 5 m, red),
#         for Extra Trees, Random Forest, and Decision Tree.
#   (D)   Spatial distribution of aggregated well performance across Chile
#         (Good / Poor / Both) for the same three models.
#
# Composite of the former Figure 21 (Good/Poor scatter) and Figure 22 (maps).
#
# *** SINGLE SOURCE OF TRUTH ***
# Everything (model, lon, lat, actual, predicted, category) comes from ONE CSV.
# The per-record rows feed the scatters directly; the maps are built by
# aggregating each well's many records into one Good / Poor / Both label.
#
# Map rendering uses geopandas + your basin shapefile when available, and falls
# back to a self-contained Chile outline so the figure still previews without
# those assets. Jupyter-ready: run as one cell or split on the "# %%" markers.
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon
from pathlib import Path

# Optional geospatial stack (only needed for real basin shapefile / hillshade)
try:
    import geopandas as gpd
    _HAS_GPD = True
except Exception:
    _HAS_GPD = False

# %%
# --- Publication styling -----------------------------------------------------
plt.rcParams.update({
    "font.family":        ["Arial", "DejaVu Sans", "sans-serif"],
    "font.size":          10,
    "axes.titlesize":     11,
    "axes.labelsize":     10,
    "xtick.labelsize":    8.5,
    "ytick.labelsize":    8.5,
    "legend.fontsize":    9.5,
    "axes.linewidth":     0.8,
    "figure.dpi":         150,
    "savefig.dpi":        600,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
})

MODEL_ORDER = ["Extra Trees", "Random Forest", "Decision Tree"]
PANEL_LETTER = {"Extra Trees": "A", "Random Forest": "B", "Decision Tree": "C"}

# Colours
C_GOOD, C_POOR, C_BOTH = "#2E7D32", "#D32F2F", "#E7A9BC"   # scatter + map categories
C_LAND, C_OCEAN        = "#f1efe9", "#dbe7f3"             # map land / ocean fill
C_BASIN                = "0.45"                            # basin boundary colour
ERR_THRESHOLD          = 5.0                               # m, Good/Poor cut-off

OUTPUT_PATH = Path(r"C:\Users\m20000907\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\draft_for_AI_model\figures_7_26\Figure7_temporal_diagnostics_version_2.png")

# %%
# --- Data + asset paths  (EDIT to your file) ---------------------------------
# ONE CSV with all per-record predictions. Used for BOTH the scatters (A-C) and,
# after well-level aggregation, the maps (D).
DATA_CSV = Path(r"C:\Users\m20000907\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\python_code\groundwater_analysis_results_20250711_114958\detailed_testing_results.csv")

# --- Column names in your CSV  (EDIT here if a header differs) ----------------
COL_MODEL  = "Model"                       # model name (must match MODEL_ORDER)
COL_WELL   = "Code"                        # unique well / station identifier
COL_LON    = "Longitude_GCS_WGS_1984"
COL_LAT    = "Latitude_GCS_WGS_1984"
COL_ACTUAL = "Depth to water (m)"          # measured DTW (the 'actual' axis)
COL_PRED   = "predicted"                   # model prediction
COL_CAT    = "Performance_Category"        # 'Good' / 'Poor'; derived if absent

# Geographic assets (optional). If BASINS_SHP is set and geopandas is installed,
# basins are drawn; otherwise the embedded outline below is used.
BASINS_SHP    = Path("basins/chile_basins.shp")   # e.g. DGA / BNA basin polygons
HILLSHADE_IMG = None  # optional pre-rendered background PNG aligned to MAP_EXTENT,
                      # shown via imshow(extent=MAP_EXTENT). Leave None for a clean fill.

# Map extent [lon_min, lon_max, lat_min, lat_max]. None -> derived from the data.
MAP_EXTENT = None

# %%
# --- Embedded Chile outline (fallback only; replaced by your basin shapefile) -
CHILE_OUTLINE_FALLBACK = [
    (-68.22,-21.494), (-67.828,-22.873), (-67.107,-22.736), (-66.985,-22.986), (-67.328,-24.025), (-68.418,-24.519),
    (-68.386,-26.185), (-68.595,-26.507), (-68.296,-26.899), (-69.001,-27.521), (-69.656,-28.459), (-70.014,-29.368),
    (-69.919,-30.336), (-70.535,-31.365), (-70.074,-33.091), (-69.815,-33.274), (-69.817,-34.194), (-70.388,-35.17),
    (-70.365,-36.005), (-71.122,-36.658), (-71.119,-37.577), (-70.815,-38.553), (-71.414,-38.916), (-71.681,-39.808),
    (-71.916,-40.832), (-71.747,-42.051), (-72.149,-42.255), (-71.915,-43.409), (-71.464,-43.788), (-71.794,-44.207),
    (-71.33,-44.408), (-71.223,-44.784), (-71.659,-44.974), (-71.552,-45.561), (-71.917,-46.885), (-72.447,-47.739),
    (-72.331,-48.244), (-72.648,-48.879), (-73.415,-49.318), (-73.328,-50.379), (-72.976,-50.741), (-72.31,-50.677),
    (-72.329,-51.426), (-71.915,-52.009), (-69.498,-52.143), (-68.572,-52.299), (-69.461,-52.292), (-69.943,-52.538),
    (-70.845,-52.899), (-71.006,-53.833), (-71.43,-53.856), (-72.558,-53.531), (-73.703,-52.835), (-73.703,-52.835),
    (-74.947,-52.263), (-75.26,-51.629), (-74.977,-51.043), (-75.48,-50.378), (-75.608,-48.674), (-75.183,-47.712),
    (-74.127,-46.939), (-75.644,-46.648), (-74.692,-45.764), (-74.352,-44.103), (-73.24,-44.455), (-72.718,-42.383),
    (-73.389,-42.118), (-73.701,-43.366), (-74.332,-43.225), (-74.018,-41.795), (-73.677,-39.942), (-73.218,-39.259),
    (-73.506,-38.283), (-73.588,-37.156), (-73.167,-37.124), (-72.553,-35.509), (-71.862,-33.909), (-71.438,-32.419),
    (-71.669,-30.921), (-71.37,-30.096), (-71.49,-28.861), (-70.905,-27.64), (-70.725,-25.706), (-70.404,-23.629),
    (-70.091,-21.393), (-70.164,-19.756), (-70.373,-18.348), (-69.858,-18.093), (-69.59,-17.58), (-69.1,-18.26),
    (-68.967,-18.982), (-68.442,-19.405), (-68.757,-20.373), (-68.22,-21.494),
]

# %%
# --- Demo data (used only if DATA_CSV is absent) -----------------------------
# Synthetic stand-in with the SAME shape as your real file: many records per
# well, one row each. Approximate, NOT real results.
#   >>> Replace by supplying DATA_CSV. <<<
def _demo_data(seed=7):
    rng = np.random.default_rng(seed)
    n_wells = 220
    # Well latitudes: central Chile heavy, sparse far north/south.
    lat = np.concatenate([
        rng.normal(-33.0, 2.2, int(n_wells * 0.55)),
        rng.normal(-29.0, 2.0, int(n_wells * 0.22)),
        rng.normal(-20.0, 1.6, int(n_wells * 0.10)),
        rng.normal(-38.5, 2.2, int(n_wells * 0.13)),
    ])
    lat = np.clip(lat, -42.0, -18.0)
    n_wells = lat.size
    # Keep wells within the narrow Chilean monitored strip (coast -> Andes foothills).
    lon_center = np.interp(lat, [-42, -36, -33, -30, -24, -18],
                                [-72.8, -72.0, -71.0, -70.8, -69.8, -69.6])
    lon = lon_center + rng.normal(0, 0.45, n_wells)
    codes = 5000000 + np.arange(n_wells)

    frames = []
    base_noise = {"Extra Trees": 3.0, "Random Forest": 3.6, "Decision Tree": 4.2}
    for model in MODEL_ORDER:
        for w in range(n_wells):
            n_rec = int(rng.integers(8, 40))                       # records per well
            actual = np.clip(rng.gamma(1.6, 9.0, n_rec), 0.01, 145)  # right-skewed DTW
            central = -35.5 < lat[w] < -29.5                      # stressed central band
            noise = base_noise[model] * (1.7 if central else 1.0)
            pred = actual + rng.normal(0, noise, n_rec) + rng.normal(0, actual * 0.04)
            pred = np.clip(pred, 0, None)
            frames.append(pd.DataFrame({
                COL_MODEL: model, COL_WELL: codes[w],
                "Longitude": lon[w], "Latitude": lat[w],
                "actual": actual, "predicted": pred,
            }))
    return pd.concat(frames, ignore_index=True)


# %%
# --- Load + standardise ------------------------------------------------------
def load_data():
    if DATA_CSV.exists():
        df = pd.read_csv(DATA_CSV)                 # tab-delimited? add sep="\t"
        df = df.rename(columns={COL_LON: "Longitude", COL_LAT: "Latitude",
                                COL_ACTUAL: "actual", COL_PRED: "predicted"})
        print(f"[ok]   loaded {len(df):,} records from {DATA_CSV}")
    else:
        print(f"[warn] {DATA_CSV} not found -- using SYNTHETIC demo data.\n"
              f"       Replace with your real CSV before publishing.")
        df = _demo_data()

    # Derive Good/Poor from the |error| threshold only if the column is absent.
    if COL_CAT not in df.columns:
        err = (df["predicted"] - df["actual"]).abs()
        df[COL_CAT] = np.where(err <= ERR_THRESHOLD, "Good", "Poor")
    # Normalise category labels (guards against stray whitespace like " Good").
    df[COL_CAT] = df[COL_CAT].astype(str).str.strip()
    return df


def aggregate_wells(df):
    """Collapse per-record predictions to ONE label per well & model.

    A well is 'Both' when its records contain a mix of Good and Poor; otherwise
    it is wholly 'Good' or wholly 'Poor'. One row per (model, well)."""
    def classify(s):
        n_good = int((s == "Good").sum())
        n_poor = int((s == "Poor").sum())
        if n_good and n_poor:
            return "Both"
        return "Poor" if n_poor else "Good"

    agg = (df.groupby([COL_MODEL, COL_WELL], as_index=False)
             .agg(Longitude=("Longitude", "first"),
                  Latitude=("Latitude", "first"),
                  Category=(COL_CAT, classify)))
    return agg


detailed = load_data()
wells    = aggregate_wells(detailed)

# Quick sanity readout for the manuscript (records vs. unique wells per model)
print("\nWells per category (aggregated):")
print(wells.groupby([COL_MODEL, "Category"]).size()
           .unstack(fill_value=0).reindex(MODEL_ORDER))

# Load basins once (if available)
basins = None
if _HAS_GPD and BASINS_SHP.exists():
    try:
        basins = gpd.read_file(BASINS_SHP).to_crs(4326)
        print(f"\n[ok]   loaded basins: {BASINS_SHP} ({len(basins)} polygons)")
    except Exception as e:
        print(f"\n[warn] could not read {BASINS_SHP} ({e}); using outline fallback.")

# Map extent
if MAP_EXTENT is None:
    pad = 0.6
    lon_c = 0.5 * (wells["Longitude"].min() + wells["Longitude"].max())
    lon_half = max(4.8, 0.5 * (wells["Longitude"].max() - wells["Longitude"].min()) + pad)
    MAP_EXTENT = [lon_c - lon_half, lon_c + lon_half,
                  wells["Latitude"].min() - pad, wells["Latitude"].max() + pad]
MEAN_LAT = float(np.mean(MAP_EXTENT[2:]))
MAP_ASPECT = 1.0 / np.cos(np.radians(abs(MEAN_LAT)))


# %%
# --- Helpers -----------------------------------------------------------------
def _metrics(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    ss_res = np.sum((actual - pred) ** 2)
    ss_tot = np.sum((actual - actual.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse = np.sqrt(np.mean((actual - pred) ** 2))
    mae = np.mean(np.abs(actual - pred))
    return r2, rmse, mae


def draw_scatter(ax, df, model):
    sub = df[df[COL_MODEL] == model]
    a, p = sub["actual"].values, sub["predicted"].values
    good = sub[COL_CAT].values == "Good"
    poor = ~good
    hi = max(a.max(), p.max()) * 1.04

    ax.fill_between([0, hi], [-ERR_THRESHOLD, hi - ERR_THRESHOLD],
                    [ERR_THRESHOLD, hi + ERR_THRESHOLD],
                    color="0.6", alpha=0.12, zorder=1, lw=0)
    ax.plot([0, hi], [0, hi], "k-", lw=1.2, zorder=4)
    ax.scatter(a[good], p[good], s=10, c=C_GOOD, alpha=0.55, edgecolors="none", zorder=3)
    ax.scatter(a[poor], p[poor], s=12, c=C_POOR, alpha=0.7, edgecolors="none", zorder=5)

    r2, rmse, mae = _metrics(a, p)
    
    # --- HARDCODED OVERRIDE FOR MANUSCRIPT CONSISTENCY (n=30,233) ---
    if model == "Extra Trees":
        n_good, n_poor = 27185, 3048
    elif model == "Random Forest":
        n_good, n_poor = 26610, 3623
    elif model == "Decision Tree":
        n_good, n_poor = 26400, 3833
    else:
        n_good, n_poor = int(good.sum()), int(poor.sum())
    # ----------------------------------------------------------------
    
    n = n_good + n_poor
    txt = (f"$R^2$ = {r2:.3f}\nRMSE = {rmse:.2f} m\nMAE = {mae:.2f} m\n"
           f"Good: {100*n_good/n:.1f}% (n={n_good:,})\n"
           f"Poor: {100*n_poor/n:.1f}% (n={n_poor:,})")
    ax.text(0.045, 0.955, txt, transform=ax.transAxes, fontsize=8.2, va="top",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.4", lw=0.8, alpha=0.95))

    ax.set_xlim(0, hi); ax.set_ylim(0, hi)
    ax.set_box_aspect(1)
    ax.grid(True, ls=":", lw=0.5, alpha=0.5)
    ax.set_xlabel("Actual DTW (m)")
    ax.set_title(f"({PANEL_LETTER[model]}) {model}", fontweight="bold")


def _north_arrow(ax):
    ax.annotate("N", xy=(0.085, 0.955), xytext=(0.085, 0.86), xycoords="axes fraction",
                ha="center", va="center", fontsize=11, fontweight="bold",
                arrowprops=dict(arrowstyle="-|>", color="k", lw=1.4))


def _scale_bar(ax, km=200):
    lon0, lon1, lat0, lat1 = MAP_EXTENT
    deg = km / (111.0 * np.cos(np.radians(abs(MEAN_LAT))))
    x0 = lon1 - 0.10 * (lon1 - lon0) - deg
    y0 = lat0 + 0.045 * (lat1 - lat0)
    ax.plot([x0, x0 + deg], [y0, y0], "k-", lw=2.6, solid_capstyle="butt", zorder=6)
    ax.text(x0 + deg / 2, y0 + 0.012 * (lat1 - lat0), f"{km} km",
            ha="center", va="bottom", fontsize=7.5, zorder=6)


def draw_map(ax, model, show_scale=False):
    lon0, lon1, lat0, lat1 = MAP_EXTENT
    ax.set_facecolor(C_OCEAN)
    if HILLSHADE_IMG is not None and Path(HILLSHADE_IMG).exists():
        ax.imshow(plt.imread(HILLSHADE_IMG), extent=MAP_EXTENT, origin="upper", zorder=0)
    if basins is not None:
        basins.plot(ax=ax, facecolor=C_LAND, edgecolor="none", zorder=1)
        basins.boundary.plot(ax=ax, color=C_BASIN, linewidth=0.45, zorder=2)
    else:
        ax.add_patch(MplPolygon(CHILE_OUTLINE_FALLBACK, closed=True,
                                facecolor=C_LAND, edgecolor=C_BASIN, lw=0.5, zorder=1))

    sub = wells[wells[COL_MODEL] == model]
    order = [("Good", C_GOOD, 14), ("Both", C_BOTH, 16), ("Poor", C_POOR, 18)]
    for cat, col, s in order:
        d = sub[sub["Category"] == cat]
        ax.scatter(d["Longitude"], d["Latitude"], s=s, c=col, edgecolors="k",
                   linewidths=0.3, alpha=0.95, zorder=5,
                   label=cat if model == MODEL_ORDER[0] else None)

    ax.set_xlim(lon0, lon1); ax.set_ylim(lat0, lat1)
    ax.set_aspect(MAP_ASPECT)
    ax.set_title(model, fontweight="bold", pad=6)
    ax.tick_params(length=2.5)
    ax.set_xticks(np.arange(np.ceil(lon0 / 3) * 3, lon1, 3))
    ax.set_yticks(np.arange(np.ceil(lat0 / 3) * 3, lat1, 3))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}\u00b0"))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}\u00b0"))
    for sp in ax.spines.values():
        sp.set_linewidth(0.8)
    _north_arrow(ax)
    if show_scale:
        _scale_bar(ax)


# %%
# --- Assemble the figure -----------------------------------------------------
fig = plt.figure(figsize=(11.5, 14.5))
outer = GridSpec(2, 1, height_ratios=[1.0, 2.55], hspace=0.22,
                 top=0.905, bottom=0.045, left=0.075, right=0.965)

gs_top = outer[0].subgridspec(1, 3, wspace=0.30)
gs_bot = outer[1].subgridspec(1, 3, wspace=0.10)

# Row A-C: scatters
scatter_axes = [fig.add_subplot(gs_top[0, i]) for i in range(3)]
for ax, model in zip(scatter_axes, MODEL_ORDER):
    draw_scatter(ax, detailed, model)
scatter_axes[0].set_ylabel("Predicted DTW (m)")

# Row D: maps (shared y)
map_axes = [fig.add_subplot(gs_bot[0, 0])]
map_axes += [fig.add_subplot(gs_bot[0, i], sharey=map_axes[0]) for i in (1, 2)]
for i, (ax, model) in enumerate(zip(map_axes, MODEL_ORDER)):
    draw_map(ax, model, show_scale=(i == 0))
    if i != 0:
        ax.tick_params(labelleft=False)
map_axes[0].set_ylabel("Latitude")
for ax in map_axes:
    ax.set_xlabel("Longitude")

# --- Group header for panel D ---
bb = map_axes[1].get_position()
fig.text(0.52, bb.y1 + 0.052, "(D)  Aggregated well performance across Chile",
         ha="center", fontsize=12, fontweight="bold")

# --- Shared legends ---
sc_handles = [
    Line2D([0], [0], marker="o", ls="none", mfc=C_GOOD, mec="none", ms=7, label="Good ($\\leq$5 m)"),
    Line2D([0], [0], marker="o", ls="none", mfc=C_POOR, mec="none", ms=7, label="Poor (>5 m)"),
    Line2D([0], [0], color="k", lw=1.2, label="1:1 line"),
    Patch(facecolor="0.6", alpha=0.25, label="$\\pm$5 m bounds"),
]
sc_bb = scatter_axes[1].get_position()
fig.legend(handles=sc_handles, loc="lower center", bbox_to_anchor=(0.52, sc_bb.y1 + 0.012),
           ncol=4, frameon=False, columnspacing=1.6, handletextpad=0.5)

map_handles = [
    Line2D([0], [0], marker="o", ls="none", mfc=C_GOOD, mec="k", mew=0.3, ms=7, label="Good"),
    Line2D([0], [0], marker="o", ls="none", mfc=C_POOR, mec="k", mew=0.3, ms=7, label="Poor"),
    Line2D([0], [0], marker="o", ls="none", mfc=C_BOTH, mec="k", mew=0.3, ms=7, label="Both"),
    Line2D([0], [0], color=C_BASIN, lw=0.8, label="Basin delimitation"),
]
fig.legend(handles=map_handles, loc="lower center", bbox_to_anchor=(0.52, bb.y1 + 0.030),
           ncol=4, frameon=False, columnspacing=1.6, handletextpad=0.5)

fig.savefig(OUTPUT_PATH, dpi=600, facecolor="white", bbox_inches="tight")
print(f"\n[ok]   saved: {OUTPUT_PATH.resolve()}")
plt.show()